# Rehabilitation Work Order Dataset – Preparation & Analysis

Contract **4235-101 / WO 3** · Gravity Sewer Rehabilitation · Contractor: PM dba IPR  
100 pipe segments · pure Python standard library

In [1]:
import openpyxl, json, csv, re, math, statistics, io
from collections import Counter, defaultdict

## 1. Load Raw Data

In [2]:
SOURCE = '/root/.claude/uploads/52187701-0c0b-4618-9d7a-7bcca86a3d8e/01f93a8e-sample_rehab_wo.xlsx'

wb = openpyxl.load_workbook(SOURCE)
ws = wb['Sheet1']
raw_rows = list(ws.iter_rows(min_row=1, max_row=101, values_only=True))

headers = raw_rows[0]
raw_data = raw_rows[1:]

print(f'Columns ({len(headers)}): {list(headers)}')
print(f'Data rows: {len(raw_data)}')

Columns (27): ['Contract_WO', 'Contract', 'WO', 'DateIssued', 'IMS', 'Upstream_MH', 'Downstream_MH', 'UFID', 'Diameter_Inch', 'Length_Ft', 'WorkType', 'Other_WO_Description', 'WorkTypeDetail', 'AssetType', 'IPS_Cancel_Status', 'Contractor', 'SiteSpecificProject', 'In_CD_Remediation', 'RFI_Reason', 'RFI_Date', 'RFI_Comment', 'RFI_NeedToReview', 'GSM1', 'GSM_R', 'In CD 4/5 List', 'Remediated?', 'Meter Basin']
Data rows: 100


## 2. Data Quality Audit

In [3]:
n = len(raw_data)
col_audit = []
for i, h in enumerate(headers):
    vals       = [r[i] for r in raw_data]
    non_null   = [v for v in vals if v is not None]
    unique_set = set(non_null)
    is_formula = any(isinstance(v, str) and v.startswith('=') for v in non_null)
    pct_fill   = round(len(non_null) / n * 100, 1)
    col_audit.append({
        'col': i, 'name': h,
        'non_null': len(non_null), 'pct_fill': pct_fill,
        'unique': len(unique_set), 'is_formula': is_formula,
        'sample': str(list(unique_set)[:3])
    })

hdr = f"{'Col':>3}  {'Field':<28}  {'Fill%':>6}  {'Unique':>6}  {'Formula?':>8}  Sample"
print(hdr)
print('-' * len(hdr))
for a in col_audit:
    flag = ' *** FORMULA' if a['is_formula'] else (' (all null)' if a['non_null'] == 0 else (''))
    print(f"{a['col']:>3}  {str(a['name']):<28}  {a['pct_fill']:>6.1f}%  {a['unique']:>6}  {str(a['is_formula']):>8}  {a['sample'][:50]}{flag}")

Col  Field                          Fill%  Unique  Formula?  Sample
-------------------------------------------------------------------
  0  Contract_WO                    100.0%       1     False  ['4235-101_3']
  1  Contract                       100.0%       1     False  ['4235-101']
  2  WO                             100.0%       1     False  [3]
  3  DateIssued                       0.0%       0     False  [] (all null)
  4  IMS                            100.0%     100     False  [14879744, 14879745, 14879746]
  5  Upstream_MH                    100.0%      99     False  ['IA064063', 'IA064001', 'IA064010']
  6  Downstream_MH                  100.0%      80     False  ['IA064063', 'IA064001', 'IA064010']
  7  UFID                           100.0%     100     False  [3199488, 3199489, 3199490]
  8  Diameter_Inch                  100.0%       8     False  [6, 8, 10]
  9  Length_Ft                      100.0%      83     False  [0, 513, 8]
 10  WorkType                       100.0%

## 3. Drop Uninformative Columns

Dropped reasons:
- **All-null**: DateIssued, SiteSpecificProject, In_CD_Remediation, all RFI_*, Remediated?, Meter Basin
- **Single value** (no variance): Contract_WO, Contract, WO, AssetType, IPS_Cancel_Status, Contractor
- **Unresolved Excel formula**: In CD 4/5 List
- **Redundant** (reverse direction of GSM1): GSM_R

In [4]:
DROP_COLS = {
    'Contract_WO', 'Contract', 'WO',          # single value
    'DateIssued',                               # 100% null
    'AssetType', 'IPS_Cancel_Status',           # single value
    'Contractor',                               # single value
    'SiteSpecificProject', 'In_CD_Remediation', # 100% null
    'RFI_Reason', 'RFI_Date',                   # 100% null
    'RFI_Comment', 'RFI_NeedToReview',          # 100% null
    'In CD 4/5 List',                           # unresolved formula
    'Remediated?', 'Meter Basin',               # 100% null
    'GSM_R',                                    # redundant reverse key
}

keep_idx  = [i for i, h in enumerate(headers) if h not in DROP_COLS]
keep_cols = [headers[i] for i in keep_idx]

clean = []
for r in raw_data:
    clean.append({keep_cols[j]: r[keep_idx[j]] for j in range(len(keep_idx))})

print(f'Kept {len(keep_cols)} columns: {keep_cols}')
print(f'Rows: {len(clean)}')

Kept 10 columns: ['IMS', 'Upstream_MH', 'Downstream_MH', 'UFID', 'Diameter_Inch', 'Length_Ft', 'WorkType', 'Other_WO_Description', 'WorkTypeDetail', 'GSM1']
Rows: 100


## 4. Feature Engineering

In [5]:
# ── 4a. Diameter category ─────────────────────────────────────────────────
def diam_cat(d):
    if d is None:  return 'Unknown'
    if d <= 8:     return 'Small (≤8")'
    if d <= 15:    return 'Medium (10–15")'
    return             'Large (≥18")'

# ── 4b. Length category ───────────────────────────────────────────────────
def len_cat(l):
    if l is None or l == 0: return 'Unknown'
    if l < 150:  return 'Short (<150 ft)'
    if l <= 300: return 'Medium (150–300 ft)'
    return           'Long (>300 ft)'

# ── 4c. Method short label ────────────────────────────────────────────────
METHOD_MAP = {
    'CTV':            'CTV Inspection',
    'Pipe Bursting':  'Pipe Bursting',
    'Cured-in-Place': 'CIPP Lining',
    'Point Repair':   'Point Repair',
}

# ── 4d. CIPP thickness (mm) extracted from WorkTypeDetail ─────────────────
def cipp_thickness(detail):
    if not detail or 'cured-in-place' not in detail.lower():
        return None
    m = re.search(r'([\d.]+)-mm', detail)
    return float(m.group(1)) if m else None

# ── 4e. PE liner OD (inches) for Pipe Bursting ────────────────────────────
def pb_liner_od(detail):
    if not detail or 'polyethylene' not in detail.lower():
        return None
    m = re.search(r'([\d.]+) inch outside diameter', detail)
    return float(m.group(1)) if m else None

# ── 4f. Area proxy (diameter × length, sq-in·ft) ─────────────────────────
def pipe_area_proxy(d, l):
    if d is None or l is None:
        return None
    return round(math.pi * (d / 24)**2 * l, 1)   # approx ft² of pipe interior wall

for row in clean:
    row['Diam_Category']    = diam_cat(row['Diameter_Inch'])
    row['Length_Category']  = len_cat(row['Length_Ft'])
    row['Method_Short']     = METHOD_MAP.get(row['WorkType'], row['WorkType'])
    row['Is_Inspection']    = row['WorkType'] == 'CTV'
    row['Is_Structural']    = row['WorkType'] in ('Pipe Bursting', 'Cured-in-Place', 'Point Repair')
    row['CIPP_Thickness_mm']= cipp_thickness(row['WorkTypeDetail'])
    row['PB_Liner_OD_in']   = pb_liner_od(row['WorkTypeDetail'])
    row['Wall_Area_ft2']    = pipe_area_proxy(row['Diameter_Inch'], row['Length_Ft'])

print(f'Added 8 engineered columns.')
print('Sample:', {k: v for k, v in list(clean[1].items())})

Added 8 engineered columns.
Sample: {'IMS': 14879974, 'Upstream_MH': 'IA064001', 'Downstream_MH': 'IA065041', 'UFID': 3465316, 'Diameter_Inch': 15, 'Length_Ft': 300, 'WorkType': 'CTV', 'Other_WO_Description': None, 'WorkTypeDetail': 'Perform cleaning and television inspection', 'GSM1': 'GSM_IA064001IA065041', 'Diam_Category': 'Medium (10–15")', 'Length_Category': 'Medium (150–300 ft)', 'Method_Short': 'CTV Inspection', 'Is_Inspection': True, 'Is_Structural': False, 'CIPP_Thickness_mm': None, 'PB_Liner_OD_in': None, 'Wall_Area_ft2': 368.2}


## 5. Summary Statistics

In [6]:
# ── 5a. Overall stats ─────────────────────────────────────────────────────
total_pipes  = len(clean)
lengths      = [r['Length_Ft'] for r in clean if r['Length_Ft']]
total_len_ft = sum(lengths)
total_len_mi = total_len_ft / 5280
mean_len     = statistics.mean(lengths)
median_len   = statistics.median(lengths)

print(f'Total pipes         : {total_pipes}')
print(f'Total length        : {total_len_ft:,.0f} ft  ({total_len_mi:.2f} miles)')
print(f'Mean pipe length    : {mean_len:.1f} ft')
print(f'Median pipe length  : {median_len:.1f} ft')
print(f'Min / Max length    : {min(lengths)} / {max(lengths)} ft')

Total pipes         : 100
Total length        : 25,093 ft  (4.75 miles)
Mean pipe length    : 253.5 ft
Median pipe length  : 279.0 ft
Min / Max length    : 8 / 523 ft


In [7]:
# ── 5b. Work-type breakdown ───────────────────────────────────────────────
wt_count  = Counter(r['WorkType'] for r in clean)
wt_len    = defaultdict(float)
for r in clean:
    wt_len[r['WorkType']] += r['Length_Ft'] or 0

print('WorkType          Count   Length(ft)  Length(mi)  Pct Count  Pct Length')
print('-' * 70)
for wt in sorted(wt_count, key=lambda x: -wt_count[x]):
    c  = wt_count[wt]
    ll = wt_len[wt]
    print(f'{wt:<18} {c:>5}   {ll:>9,.0f}  {ll/5280:>9.2f}  {c/total_pipes*100:>8.1f}%  {ll/total_len_ft*100:>8.1f}%')

WorkType          Count   Length(ft)  Length(mi)  Pct Count  Pct Length
----------------------------------------------------------------------
Pipe Bursting         71      16,044       3.04      71.0%      63.9%
Cured-in-Place        18       4,956       0.94      18.0%      19.8%
CTV                   10       3,804       0.72      10.0%      15.2%
Point Repair           1         289       0.05       1.0%       1.2%


In [8]:
# ── 5c. Diameter breakdown ────────────────────────────────────────────────
d_count = Counter(r['Diameter_Inch'] for r in clean)
d_len   = defaultdict(float)
for r in clean:
    d_len[r['Diameter_Inch']] += r['Length_Ft'] or 0

print('Diam(in)  Count   Length(ft)  Pct Count  Pct Length')
print('-' * 53)
for d in sorted(d_count):
    c   = d_count[d]
    ll  = d_len[d]
    dq  = str(d) + '"'
    print(f'{dq:>8}  {c:>5}   {ll:>9,.0f}  {c/total_pipes*100:>8.1f}%  {ll/total_len_ft*100:>8.1f}%')

Diam(in)  Count   Length(ft)  Pct Count  Pct Length
-----------------------------------------------------
      6"     12       1,453      12.0%       5.8%
      8"     57      14,793      57.0%      59.0%
     10"      5       1,692       5.0%       6.7%
     12"      7       2,089       7.0%       8.3%
     15"      3         686       3.0%       2.7%
     18"      1          25       1.0%       0.1%
     21"      3         764       3.0%       3.0%
     30"     12       3,591      12.0%      14.3%


In [9]:
# ── 5d. Work type × Diameter cross-tab (pipe count) ──────────────────────
wt_list   = sorted(wt_count.keys())
diam_list = sorted(d_count.keys())
xtab = defaultdict(lambda: defaultdict(int))
xtab_len = defaultdict(lambda: defaultdict(float))
for r in clean:
    xtab[r['WorkType']][r['Diameter_Inch']] += 1
    xtab_len[r['WorkType']][r['Diameter_Inch']] += r['Length_Ft'] or 0

print('Cross-tab: WorkType vs Diameter (pipe count)')
header_row = '{:<20}'.format('WorkType') + ''.join('{:>8}'.format(str(d) + '"') for d in diam_list)
print(header_row)
for wt in wt_list:
    row_str = '{:<20}'.format(wt) + ''.join('{:>8}'.format(xtab[wt][d]) for d in diam_list)
    print(row_str)

Cross-tab: WorkType vs Diameter (pipe count)
WorkType                  6"      8"     10"     12"     15"     18"     21"     30"
CTV                        0       7       0       2       1       0       0       0
Cured-in-Place             0       0       1       0       1       1       3      12
Pipe Bursting             12      49       4       5       1       0       0       0
Point Repair               0       1       0       0       0       0       0       0


In [10]:
# ── 5e. Diameter category breakdown ──────────────────────────────────────
dc_count = Counter(r['Diam_Category'] for r in clean)
dc_len   = defaultdict(float)
for r in clean:
    dc_len[r['Diam_Category']] += r['Length_Ft'] or 0

print('Diameter Category      Count  Length(ft)')
for k in ['Small (≤8")', 'Medium (10–15")', 'Large (≥18")', 'Unknown']:
    if k in dc_count:
        print(f'{k:<24}  {dc_count[k]:>4}  {dc_len[k]:>9,.0f}')

Diameter Category      Count  Length(ft)
Small (≤8")                 69     16,246
Medium (10–15")             15      4,467
Large (≥18")                16      4,380


In [11]:
# ── 5f. Length distribution (histogram buckets) ───────────────────────────
BUCKETS = [(0,100),(100,200),(200,300),(300,400),(400,500),(500,600)]
bucket_counts = {b: 0 for b in BUCKETS}
for r in clean:
    l = r['Length_Ft'] or 0
    for lo, hi in BUCKETS:
        if lo <= l < hi:
            bucket_counts[(lo, hi)] += 1
            break

print('Length bucket  Count  Bar')
for (lo, hi), c in bucket_counts.items():
    bar = '█' * c
    print(f'{lo:>4}–{hi:<4} ft  {c:>4}  {bar}')

Length bucket  Count  Bar
   0–100  ft    21  █████████████████████
 100–200  ft     9  █████████
 200–300  ft    27  ███████████████████████████
 300–400  ft    30  ██████████████████████████████
 400–500  ft    11  ███████████
 500–600  ft     2  ██


In [12]:
# ── 5g. CIPP thickness summary ────────────────────────────────────────────
cipp_rows = [r for r in clean if r['CIPP_Thickness_mm'] is not None]
thick_counter = Counter(r['CIPP_Thickness_mm'] for r in cipp_rows)
print('CIPP Thickness (mm) distribution:')
for t, c in sorted(thick_counter.items()):
    print(f'  {t} mm : {c} pipes')

pb_rows = [r for r in clean if r['PB_Liner_OD_in'] is not None]
liner_counter = Counter(r['PB_Liner_OD_in'] for r in pb_rows)
print('\nPipe Bursting liner OD (in) distribution:')
for t, c in sorted(liner_counter.items()):
    print(f'  {t}" OD : {c} pipes')

CIPP Thickness (mm) distribution:
  6.0 mm : 1 pipes
  10.5 mm : 1 pipes
  13.5 mm : 1 pipes
  15.0 mm : 3 pipes
  21.0 mm : 12 pipes

Pipe Bursting liner OD (in) distribution:
  8.625" OD : 61 pipes
  10.75" OD : 4 pipes
  12.75" OD : 5 pipes
  16.0" OD : 1 pipes


## 6. Data Quality Flags

In [13]:
flags = []

# Zero-length pipes
zero_len = [r for r in clean if r['Length_Ft'] == 0]
if zero_len:
    flags.append({'flag': 'Zero length', 'count': len(zero_len),
                  'detail': [r['IMS'] for r in zero_len]})

# Very short (< 20 ft) non-zero pipes
short = [r for r in clean if r['Length_Ft'] and 0 < r['Length_Ft'] < 20]
if short:
    flags.append({'flag': 'Very short (<20 ft)', 'count': len(short),
                  'detail': [(r['IMS'], r['Length_Ft']) for r in short]})

# Missing Other_WO_Description for structural work
missing_desc = [r for r in clean if r['Is_Structural'] and not r.get('Other_WO_Description')]
if missing_desc:
    flags.append({'flag': 'Structural WO missing Other_WO_Description',
                  'count': len(missing_desc),
                  'detail': [r['IMS'] for r in missing_desc][:5]})

# CIPP on small pipes (unusual)
cipp_small = [r for r in clean
              if r['WorkType'] == 'Cured-in-Place' and r['Diameter_Inch'] and r['Diameter_Inch'] < 15]
if cipp_small:
    flags.append({'flag': 'CIPP on pipe < 15" (verify spec)',
                  'count': len(cipp_small),
                  'detail': [(r['IMS'], r['Diameter_Inch']) for r in cipp_small][:5]})

print(f'Data quality flags: {len(flags)}')
for f in flags:
    print(f"  [{f['flag']}] — {f['count']} occurrences | {f['detail']}")

Data quality flags: 4
  [Zero length] — 1 occurrences | [14879779]
  [Very short (<20 ft)] — 3 occurrences | [(14948101, 8), (14948105, 16), (14950133, 11)]
  [Structural WO missing Other_WO_Description] — 30 occurrences | [14879781, 14879955, 14879954, 14879953, 14879783]
  [CIPP on pipe < 15" (verify spec)] — 1 occurrences | [(14950128, 10)]


## 7. Export Cleaned Dataset + Dashboard Payload

In [14]:
# ── 7a. Cleaned dataset → CSV ─────────────────────────────────────────────
csv_cols = [
    'IMS','UFID','Upstream_MH','Downstream_MH',
    'Diameter_Inch','Length_Ft','WorkType','Method_Short',
    'WorkTypeDetail','Other_WO_Description','GSM1',
    'Diam_Category','Length_Category','Is_Inspection','Is_Structural',
    'CIPP_Thickness_mm','PB_Liner_OD_in','Wall_Area_ft2',
]
with open('rehab_wo_clean.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=csv_cols, extrasaction='ignore')
    writer.writeheader()
    writer.writerows(clean)
print('Exported rehab_wo_clean.csv')

# ── 7b. Dashboard payload → JSON ──────────────────────────────────────────
wt_breakdown = [
    {'work_type': wt,
     'method_short': METHOD_MAP.get(wt, wt),
     'count': wt_count[wt],
     'pct_count': round(wt_count[wt] / total_pipes * 100, 1),
     'total_length_ft': round(wt_len[wt], 0),
     'pct_length': round(wt_len[wt] / total_len_ft * 100, 1)}
    for wt in sorted(wt_count, key=lambda x: -wt_count[x])
]

diam_breakdown = [
    {'diameter_in': d,
     'count': d_count[d],
     'total_length_ft': round(d_len[d], 0),
     'pct_count': round(d_count[d] / total_pipes * 100, 1)}
    for d in sorted(d_count)
]

xtab_export = {
    'work_types': wt_list,
    'diameters': diam_list,
    'count_matrix': [[xtab[wt][d] for d in diam_list] for wt in wt_list],
    'length_matrix': [[round(xtab_len[wt][d], 0) for d in diam_list] for wt in wt_list],
}

hist_data = [
    {'label': f'{lo}–{hi} ft', 'lo': lo, 'hi': hi, 'count': c}
    for (lo, hi), c in bucket_counts.items()
]

diam_cat_data = [
    {'category': k, 'count': dc_count.get(k, 0), 'total_length_ft': round(dc_len.get(k, 0), 0)}
    for k in ['Small (≤8")', 'Medium (10–15")', 'Large (≥18")']
]

table_rows = []
for r in clean:
    table_rows.append({
        'ims': r['IMS'], 'ufid': r['UFID'],
        'us_mh': r['Upstream_MH'], 'ds_mh': r['Downstream_MH'],
        'diam': r['Diameter_Inch'], 'length': r['Length_Ft'],
        'work_type': r['WorkType'], 'method_short': r['Method_Short'],
        'diam_cat': r['Diam_Category'], 'len_cat': r['Length_Category'],
        'cipp_mm': r['CIPP_Thickness_mm'], 'pb_od': r['PB_Liner_OD_in'],
        'gsm': r['GSM1'],
    })

dashboard_payload = {
    'meta': {
        'contract': '4235-101', 'wo': 3,
        'contractor': 'PM dba IPR',
        'asset_type': 'Gravity Main',
        'total_pipes': total_pipes,
        'total_length_ft': round(total_len_ft, 0),
        'total_length_mi': round(total_len_mi, 2),
        'mean_length_ft': round(mean_len, 1),
        'median_length_ft': float(median_len),
        'min_length_ft': min(lengths),
        'max_length_ft': max(lengths),
        'structural_pipes': sum(1 for r in clean if r['Is_Structural']),
        'inspection_pipes': sum(1 for r in clean if r['Is_Inspection']),
    },
    'wt_breakdown': wt_breakdown,
    'diam_breakdown': diam_breakdown,
    'diam_cat_data': diam_cat_data,
    'xtab': xtab_export,
    'histogram': hist_data,
    'data_quality': {'flags': flags},
    'table': table_rows,
}

with open('rehab_dashboard_data.json', 'w') as f:
    json.dump(dashboard_payload, f, indent=2, default=str)

print('Exported rehab_dashboard_data.json')

Exported rehab_wo_clean.csv
Exported rehab_dashboard_data.json


In [15]:
# ── Final summary printout ────────────────────────────────────────────────
print('='*60)
print('REHAB WORK ORDER — DATASET SUMMARY')
print('='*60)
print(f'Contract / WO        : 4235-101 / WO 3')
print(f'Contractor           : PM dba IPR')
print(f'Total pipe segments  : {total_pipes}')
print(f'Total length         : {total_len_ft:,.0f} ft ({total_len_mi:.2f} mi)')
print(f'Structural rehab     : {sum(1 for r in clean if r["Is_Structural"])} pipes')
print(f'CTV inspection       : {sum(1 for r in clean if r["Is_Inspection"])} pipes')
print(f'Diameter range       : 6" – 30"')
print(f'Length range         : {min(lengths)} – {max(lengths)} ft')
print(f'Unique upstream MHs  : {len(set(r["Upstream_MH"] for r in clean))}')
print(f'Unique downstream MHs: {len(set(r["Downstream_MH"] for r in clean))}')
print('='*60)

REHAB WORK ORDER — DATASET SUMMARY
Contract / WO        : 4235-101 / WO 3
Contractor           : PM dba IPR
Total pipe segments  : 100
Total length         : 25,093 ft (4.75 mi)
Structural rehab     : 90 pipes
CTV inspection       : 10 pipes
Diameter range       : 6" – 30"
Length range         : 8 – 523 ft
Unique upstream MHs  : 99
Unique downstream MHs: 80
